In [1]:
import os

from dotenv import load_dotenv

# Document Loader
from langchain_community.document_loaders import WebBaseLoader

# Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Vector Store
from langchain_community.vectorstores import FAISS

# LLM
from langchain_groq import ChatGroq

C:\Users\saini\AppData\Local\Temp\ipykernel_25036\3982319581.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
load_dotenv()

os.environ["USER_AGENT"] = "News-RAG-Chatbot"

groq_api = os.getenv("GROQ_API_KEY")

print("Groq API Loaded Successfully ✅")

Groq API Loaded Successfully ✅


# 📰 Step 3: Load BBC News Website

In this step, we load the BBC News homepage using LangChain's WebBaseLoader and inspect the extracted content before preprocessing.

In [3]:
url = "https://www.bbc.com/news"

loader = WebBaseLoader(url)

documents = loader.load()

print(f"Total Documents : {len(documents)}")

Total Documents : 1


In [4]:
print(documents[0].metadata)

{'source': 'https://www.bbc.com/news', 'title': 'BBC News - Breaking news, video and the latest top stories from the U.S. and around the world', 'description': 'Visit BBC News for the latest news, breaking news, video, audio and analysis. BBC News provides trusted World, U.S. and U.K. news as well as local and regional perspectives. Also entertainment, climate, business, science, technology and health news.', 'language': 'en-GB'}


In [5]:
print(documents[0].page_content[:1500])

BBC News - Breaking news, video and the latest top stories from the U.S. and around the worldSkip to contentBritish Broadcasting CorporationHomeNewsFootball 2026SportBusinessTechnologyHealthCultureArtsTravelEarthAudioVideoLiveUS & CanadaUKAfricaAsiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesBBC InDepthBBC VerifyHomeNewsUS & CanadaUKUK PoliticsEnglandN. IrelandN. Ireland PoliticsScotlandScotland PoliticsWalesWales PoliticsAfricaAsiaChinaIndiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesBBC InDepthBBC VerifyFootball 2026SportBusinessWorld of BusinessTechnology of BusinessNYSE Opening BellTechnologyArtificial IntelligenceIntelligence RevolutionAI v the MindTech NowHealthCultureFilm & TVMusicArt & DesignStyleBooksEntertainment NewsArtsArts in MotionTravelDestinationsAfricaAntarcticaAsiaAustralia and PacificCaribbean & BermudaCentral AmericaEuropeMiddle EastNorth AmericaSouth AmericaWorld’s TableCulture & ExperiencesAdventuresThe SpeciaListEarthScienceNatural WondersClimate Solu

In [6]:
documents[0].metadata

{'source': 'https://www.bbc.com/news',
 'title': 'BBC News - Breaking news, video and the latest top stories from the U.S. and around the world',
 'description': 'Visit BBC News for the latest news, breaking news, video, audio and analysis. BBC News provides trusted World, U.S. and U.K. news as well as local and regional perspectives. Also entertainment, climate, business, science, technology and health news.',
 'language': 'en-GB'}

In [7]:
print(documents[0].page_content[:2000])

BBC News - Breaking news, video and the latest top stories from the U.S. and around the worldSkip to contentBritish Broadcasting CorporationHomeNewsFootball 2026SportBusinessTechnologyHealthCultureArtsTravelEarthAudioVideoLiveUS & CanadaUKAfricaAsiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesBBC InDepthBBC VerifyHomeNewsUS & CanadaUKUK PoliticsEnglandN. IrelandN. Ireland PoliticsScotlandScotland PoliticsWalesWales PoliticsAfricaAsiaChinaIndiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesBBC InDepthBBC VerifyFootball 2026SportBusinessWorld of BusinessTechnology of BusinessNYSE Opening BellTechnologyArtificial IntelligenceIntelligence RevolutionAI v the MindTech NowHealthCultureFilm & TVMusicArt & DesignStyleBooksEntertainment NewsArtsArts in MotionTravelDestinationsAfricaAntarcticaAsiaAustralia and PacificCaribbean & BermudaCentral AmericaEuropeMiddle EastNorth AmericaSouth AmericaWorld’s TableCulture & ExperiencesAdventuresThe SpeciaListEarthScienceNatural WondersClimate Solu

In [8]:
content = documents[0].page_content

print("Characters :", len(content))
print("Words :", len(content.split()))

Characters : 10798
Words : 1463


In [9]:
type(documents[0])

langchain_core.documents.base.Document

In [10]:
dir(documents[0])

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_extra_info__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_on_complete__',
 '__pydantic_parent_namespace__',
 '__pydantic_post_init__',
 '__pydantic_private__',
 '__pydantic_root_model__

# ✂️ Step 4: Document Chunking

Large Language Models have context window limitations. Instead of embedding the entire webpage as a single document, we split it into smaller overlapping chunks.

Benefits:
- Better semantic search
- Higher retrieval accuracy
- Reduced token usage
- Improved answer quality

In [11]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

In [12]:
chunks = splitter.split_documents(documents)

print(f"Total Chunks Created : {len(chunks)}")

Total Chunks Created : 14


In [13]:
print(chunks[0].page_content)

BBC News - Breaking news, video and the latest top stories from the U.S. and around the worldSkip to contentBritish Broadcasting CorporationHomeNewsFootball 2026SportBusinessTechnologyHealthCultureArtsTravelEarthAudioVideoLiveUS & CanadaUKAfricaAsiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesBBC InDepthBBC VerifyHomeNewsUS & CanadaUKUK PoliticsEnglandN. IrelandN. Ireland PoliticsScotlandScotland PoliticsWalesWales PoliticsAfricaAsiaChinaIndiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesBBC InDepthBBC VerifyFootball 2026SportBusinessWorld of BusinessTechnology of BusinessNYSE Opening BellTechnologyArtificial IntelligenceIntelligence RevolutionAI v the MindTech NowHealthCultureFilm & TVMusicArt & DesignStyleBooksEntertainment NewsArtsArts in MotionTravelDestinationsAfricaAntarcticaAsiaAustralia and PacificCaribbean & BermudaCentral AmericaEuropeMiddle EastNorth AmericaSouth AmericaWorld’s TableCulture & ExperiencesAdventuresThe SpeciaListEarthScienceNatural WondersClimate


In [14]:
chunks[0].metadata

{'source': 'https://www.bbc.com/news',
 'title': 'BBC News - Breaking news, video and the latest top stories from the U.S. and around the world',
 'description': 'Visit BBC News for the latest news, breaking news, video, audio and analysis. BBC News provides trusted World, U.S. and U.K. news as well as local and regional perspectives. Also entertainment, climate, business, science, technology and health news.',
 'language': 'en-GB'}

In [15]:
for i in range(3):
    print(f"\nChunk {i+1}")
    print("-" * 50)
    print(f"Characters : {len(chunks[i].page_content)}")
    print(chunks[i].page_content[:300])


Chunk 1
--------------------------------------------------
Characters : 995
BBC News - Breaking news, video and the latest top stories from the U.S. and around the worldSkip to contentBritish Broadcasting CorporationHomeNewsFootball 2026SportBusinessTechnologyHealthCultureArtsTravelEarthAudioVideoLiveUS & CanadaUKAfricaAsiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesB

Chunk 2
--------------------------------------------------
Characters : 994
and PacificCaribbean & BermudaCentral AmericaEuropeMiddle EastNorth AmericaSouth AmericaWorld’s TableCulture & ExperiencesAdventuresThe SpeciaListEarthScienceNatural WondersClimate SolutionsSustainable BusinessGreen LivingAudioPodcast CategoriesRadioAudio FAQsVideoBBC MaestroDiscover the WorldLiveLi

Chunk 3
--------------------------------------------------
Characters : 996
incomprehensible and unjustifiable".3 hrs agoWorld CupChina tests missile in the Pacific hours after Australia-Fiji alliance signedThe test launch came hours after Aus

In [16]:
type(chunks[0])

langchain_core.documents.base.Document

In [17]:
print(f"Original Documents : {len(documents)}")
print(f"Chunks Created     : {len(chunks)}")

Original Documents : 1
Chunks Created     : 14


# 🧠 Step 5: Generate Text Embeddings

Embeddings convert text into dense numerical vectors that capture semantic meaning.

Instead of matching exact keywords, semantic search compares vector similarity, allowing the system to retrieve contextually relevant information.

In [18]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully ✅")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully ✅


In [19]:
sample_text = chunks[0].page_content

vector = embedding_model.embed_query(sample_text)

print(type(vector))

<class 'list'>


In [20]:
print("Embedding Dimension:", len(vector))

Embedding Dimension: 384


In [21]:
vector[:15]

[0.05180592089891434,
 -0.09453608095645905,
 0.010052032768726349,
 -0.06614916771650314,
 0.07227117568254471,
 0.07911179214715958,
 -0.003302032360807061,
 -0.045856770128011703,
 -0.06018826365470886,
 0.043568648397922516,
 -0.0483039990067482,
 -0.029440078884363174,
 0.0027884854935109615,
 -0.007100337650626898,
 -0.02628009393811226]

In [22]:
sentence1 = "Artificial Intelligence is transforming healthcare."

sentence2 = "AI is changing the medical industry."

sentence3 = "The football match was exciting."

embedding1 = embedding_model.embed_query(sentence1)
embedding2 = embedding_model.embed_query(sentence2)
embedding3 = embedding_model.embed_query(sentence3)

In [23]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_1_2 = cosine_similarity([embedding1], [embedding2])[0][0]
similarity_1_3 = cosine_similarity([embedding1], [embedding3])[0][0]

print(f"Similarity (1 vs 2): {similarity_1_2:.3f}")
print(f"Similarity (1 vs 3): {similarity_1_3:.3f}")

Similarity (1 vs 2): 0.800
Similarity (1 vs 3): 0.048


In [24]:
print("Chunks Ready for Vector Database:", len(chunks))

Chunks Ready for Vector Database: 14


# 🗂️ Step 6: Create FAISS Vector Database

FAISS (Facebook AI Similarity Search) is a high-performance library used to store and search dense vectors efficiently.

Instead of searching raw text, FAISS searches vector representations, enabling fast semantic retrieval.

In [25]:
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("✅ FAISS Vector Store Created Successfully")

✅ FAISS Vector Store Created Successfully


In [26]:
type(vector_store)

langchain_community.vectorstores.faiss.FAISS

In [27]:
print("Total Indexed Chunks:", vector_store.index.ntotal)

Total Indexed Chunks: 14


In [28]:
vector_store.save_local("../faiss_index")

In [29]:
loaded_vector_store = FAISS.load_local(
    "../faiss_index",
    embeddings=embedding_model,
    allow_dangerous_deserialization=True
)

print("✅ FAISS Index Loaded Successfully")

✅ FAISS Index Loaded Successfully


In [30]:
print("Indexed Chunks:", loaded_vector_store.index.ntotal)

Indexed Chunks: 14


In [31]:
query = "What is today's top news?"

results = loaded_vector_store.similarity_search(
    query=query,
    k=3
)

print(f"Retrieved {len(results)} documents")

Retrieved 3 documents


In [32]:
for i, doc in enumerate(results, start=1):
    print(f"\n{'='*60}")
    print(f"Result {i}")
    print(f"{'='*60}")
    print(doc.page_content[:500])


Result 1
BBC News - Breaking news, video and the latest top stories from the U.S. and around the worldSkip to contentBritish Broadcasting CorporationHomeNewsFootball 2026SportBusinessTechnologyHealthCultureArtsTravelEarthAudioVideoLiveUS & CanadaUKAfricaAsiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesBBC InDepthBBC VerifyHomeNewsUS & CanadaUKUK PoliticsEnglandN. IrelandN. Ireland PoliticsScotlandScotland PoliticsWalesWales PoliticsAfricaAsiaChinaIndiaAustraliaEuropeLatin AmericaMiddle EastIn Pictur

Result 2
referees at the World Cup "are just not good enough" after the dramatic last-16 win over Mexico.Most watched1Why dangerous teen sex choking is raising alarm2BBC confronts head of Meta in India over child sex abuse ads3Why Ukrainian strikes on annexed Crimea are a blow to Putin 4Watch: Why Australia's PM apologised for 'inappropriate' Kylie comments 5Watch fans react to every goal of England v Mexico gameAlso in newsChinese underground church figure Jin Mingri freed from pris

In [33]:
for i, doc in enumerate(results, start=1):
    print(f"\nResult {i} Metadata")
    print(doc.metadata)


Result 1 Metadata
{'source': 'https://www.bbc.com/news', 'title': 'BBC News - Breaking news, video and the latest top stories from the U.S. and around the world', 'description': 'Visit BBC News for the latest news, breaking news, video, audio and analysis. BBC News provides trusted World, U.S. and U.K. news as well as local and regional perspectives. Also entertainment, climate, business, science, technology and health news.', 'language': 'en-GB'}

Result 2 Metadata
{'source': 'https://www.bbc.com/news', 'title': 'BBC News - Breaking news, video and the latest top stories from the U.S. and around the world', 'description': 'Visit BBC News for the latest news, breaking news, video, audio and analysis. BBC News provides trusted World, U.S. and U.K. news as well as local and regional perspectives. Also entertainment, climate, business, science, technology and health news.', 'language': 'en-GB'}

Result 3 Metadata
{'source': 'https://www.bbc.com/news', 'title': 'BBC News - Breaking news, 

# 🔍 Step 7: Create the Retriever

A retriever is responsible for fetching the most relevant document chunks from the vector database.

Instead of manually performing similarity search, we use LangChain's retriever interface, which can later be plugged directly into a RAG chain.

In [34]:
retriever = loaded_vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Retriever created successfully ✅")

Retriever created successfully ✅


In [35]:
type(retriever)

langchain_core.vectorstores.base.VectorStoreRetriever

In [36]:
query = "What are the latest news headlines?"

retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} documents")

Retrieved 3 documents


In [37]:
for i, doc in enumerate(retrieved_docs, start=1):
    print("=" * 70)
    print(f"Result {i}")
    print("=" * 70)
    print(doc.page_content[:500])
    print()

Result 1
referees at the World Cup "are just not good enough" after the dramatic last-16 win over Mexico.Most watched1Why dangerous teen sex choking is raising alarm2BBC confronts head of Meta in India over child sex abuse ads3Why Ukrainian strikes on annexed Crimea are a blow to Putin 4Watch: Why Australia's PM apologised for 'inappropriate' Kylie comments 5Watch fans react to every goal of England v Mexico gameAlso in newsChinese underground church figure Jin Mingri freed from prisonThe Zion Church fou

Result 2
BBC News - Breaking news, video and the latest top stories from the U.S. and around the worldSkip to contentBritish Broadcasting CorporationHomeNewsFootball 2026SportBusinessTechnologyHealthCultureArtsTravelEarthAudioVideoLiveUS & CanadaUKAfricaAsiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesBBC InDepthBBC VerifyHomeNewsUS & CanadaUKUK PoliticsEnglandN. IrelandN. Ireland PoliticsScotlandScotland PoliticsWalesWales PoliticsAfricaAsiaChinaIndiaAustraliaEuropeLatin America

In [38]:
for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\nDocument {i} Metadata")
    print(doc.metadata)


Document 1 Metadata
{'source': 'https://www.bbc.com/news', 'title': 'BBC News - Breaking news, video and the latest top stories from the U.S. and around the world', 'description': 'Visit BBC News for the latest news, breaking news, video, audio and analysis. BBC News provides trusted World, U.S. and U.K. news as well as local and regional perspectives. Also entertainment, climate, business, science, technology and health news.', 'language': 'en-GB'}

Document 2 Metadata
{'source': 'https://www.bbc.com/news', 'title': 'BBC News - Breaking news, video and the latest top stories from the U.S. and around the world', 'description': 'Visit BBC News for the latest news, breaking news, video, audio and analysis. BBC News provides trusted World, U.S. and U.K. news as well as local and regional perspectives. Also entertainment, climate, business, science, technology and health news.', 'language': 'en-GB'}

Document 3 Metadata
{'source': 'https://www.bbc.com/news', 'title': 'BBC News - Breaking 

In [39]:
retriever_5 = loaded_vector_store.as_retriever(
    search_kwargs={"k": 5}
)

docs = retriever_5.invoke("Technology news")

print("Retrieved Documents:", len(docs))

Retrieved Documents: 5


In [40]:
manual_results = loaded_vector_store.similarity_search(
    "Technology news",
    k=3
)

retriever_results = retriever.invoke("Technology news")

print("Manual Search:", len(manual_results))
print("Retriever:", len(retriever_results))

Manual Search: 3
Retriever: 3


# 🤖 Step 8: Build the Retrieval-Augmented Generation (RAG) Chain

In this step we combine:

- Retriever
- Prompt Template
- Groq LLM

The retriever fetches relevant context from the vector database, and the LLM generates answers grounded in that retrieved information.

In [41]:
from langchain_core.prompts import ChatPromptTemplate

from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)

from langchain_classic.chains.retrieval import (
    create_retrieval_chain,
)

In [42]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("Groq LLM Initialized Successfully ✅")

Groq LLM Initialized Successfully ✅


In [43]:
prompt = ChatPromptTemplate.from_template("""
You are an intelligent AI News Assistant.

Answer the user's question ONLY using the provided context.

If the answer is not available in the context, reply:

"I couldn't find this information in the loaded news articles."

Context:
{context}

Question:
{input}

Answer:
""")

In [44]:
document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt
)

print("Document Chain Created Successfully ✅")

Document Chain Created Successfully ✅


In [45]:
rag_chain = create_retrieval_chain(
    retriever,
    document_chain
)

print("RAG Chain Created Successfully ✅")

RAG Chain Created Successfully ✅


In [46]:
response = rag_chain.invoke(
    {
        "input": "What is today's top news?"
    }
)

In [47]:
print(response["answer"])

I couldn't find this information in the loaded news articles.


In [48]:
response.keys()

dict_keys(['input', 'context', 'answer'])

In [49]:
len(response["context"])

3

In [50]:
for i, doc in enumerate(response["context"], start=1):
    print("=" * 80)
    print(f"Document {i}")
    print("=" * 80)
    print(doc.page_content[:500])
    print()

Document 1
BBC News - Breaking news, video and the latest top stories from the U.S. and around the worldSkip to contentBritish Broadcasting CorporationHomeNewsFootball 2026SportBusinessTechnologyHealthCultureArtsTravelEarthAudioVideoLiveUS & CanadaUKAfricaAsiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesBBC InDepthBBC VerifyHomeNewsUS & CanadaUKUK PoliticsEnglandN. IrelandN. Ireland PoliticsScotlandScotland PoliticsWalesWales PoliticsAfricaAsiaChinaIndiaAustraliaEuropeLatin AmericaMiddle EastIn Pictur

Document 2
referees at the World Cup "are just not good enough" after the dramatic last-16 win over Mexico.Most watched1Why dangerous teen sex choking is raising alarm2BBC confronts head of Meta in India over child sex abuse ads3Why Ukrainian strikes on annexed Crimea are a blow to Putin 4Watch: Why Australia's PM apologised for 'inappropriate' Kylie comments 5Watch fans react to every goal of England v Mexico gameAlso in newsChinese underground church figure Jin Mingri freed from p

In [51]:
questions = [
    "What is today's biggest news?",
    "What happened in sports?",
    "Any business news?",
    "What is happening in technology?"
]

for question in questions:

    print("\n" + "=" * 80)
    print("Question:", question)
    print("=" * 80)

    response = rag_chain.invoke({"input": question})

    print(response["answer"])


Question: What is today's biggest news?
I couldn't find this information in the loaded news articles.

Question: What happened in sports?
In sports, several events occurred: 

1. England won against Mexico in the World Cup last-16, but head coach Thomas Tuchel criticized the referees, saying they "are just not good enough".
2. Jordan Henderson injured his wrist while celebrating England's win over Mexico.
3. Norway knocked out Brazil in the World Cup, with Erling Haaland scoring both goals, and Neymar announced his international career for Brazil is "over" after the defeat.
4. Novak Djokovic broke Federer's Wimbledon record to reach the quarter-finals by beating qualifier Roman Safiullin.

Question: Any business news?
There are a few business-related news articles mentioned, including:

* A China bubble tea firm being ordered to pay Louis Vuitton $1.5m
* A global hub for fake luxury goods in Vietnam cracking down on its black market
* The NYSE Opening Bell and Technology of Business s

In [ ]:
while True:

    question = input("\nAsk News > ")

    if question.lower() == "exit":
        break

    response = rag_chain.invoke(
        {
            "input": question
        }
    )

    print("\nAnswer:\n")
    print(response["answer"])


Answer:

I couldn't find this information in the loaded news articles.

Answer:

Here are some sports news:

* Referees at the World Cup "are just not good enough" after the dramatic last-16 win over Mexico.
* Djokovic breaks Federer's Wimbledon record to reach quarters, claiming the all-time record for most men's singles match wins at Wimbledon.
* Integrity of game at stake over Fifa Balogun decision - Uefa, with Uefa saying Fifa's decision not to uphold Folarin Balogun's immediate ban at the World Cup is "unprecedented, incomprehensible and unjustifiable".

Answer:

In India, an anti-sacrilege law in Punjab has sparked controversy, reviving one of the state's most sensitive political and religious issues ahead of state elections next year. Additionally, the Indian government has ordered Meta to remove ads promoting child sexual abuse. Thousands of forgotten Punjabi WW1 soldiers are also being recognised for the first time, with nearly 10,000 soldiers' names being added to the offici

# Step 9: Extract BBC Article Links

In [1]:
import sys

sys.path.append("../src")

In [ ]:
from ingestion.link_extractor import main

main()

Total Articles : 21
Article links saved successfully.


# 🌐 Step 10: Load BBC News Articles

This step visits every extracted BBC article URL, downloads the webpage, extracts the title and article content, and stores the results in a structured JSON file.

In [7]:
import sys
sys.path.append("../src")

import importlib
import ingestion.article_loader as loader

importlib.reload(loader)

loader.main()

[1/21] https://www.bbc.com/news/articles/cy4e1lzwgvxo
[2/21] https://www.bbc.com/news/articles/cql11969g2qo
[3/21] https://www.bbc.com/news/articles/cdx7zzywz7wo
[4/21] https://www.bbc.com/news/articles/clyeele4leeo
[5/21] https://www.bbc.com/news/articles/c1jyydr7jnjo
[6/21] https://www.bbc.com/news/articles/cewqqnd7zdwo
[7/21] https://www.bbc.com/news/articles/c1my27lyjx9o
[8/21] https://www.bbc.com/news/articles/cr477en6w3go
[9/21] https://www.bbc.com/news/articles/c4gy51ky75vo
[10/21] https://www.bbc.com/news/articles/c77yykl87yno
[11/21] https://www.bbc.com/news/articles/c8e22dnl0nko
[12/21] https://www.bbc.com/news/articles/cevllyj9vv3o
[13/21] https://www.bbc.com/news/articles/c9922x87nj8o
[14/21] https://www.bbc.com/news/articles/cj4gy9qy7vyo
[15/21] https://www.bbc.com/news/articles/crlw1xrzd8yo
[16/21] https://www.bbc.com/news/articles/cwykk095yzzo
[17/21] https://www.bbc.com/news/articles/c20yy4r4g64o
[18/21] https://www.bbc.com/news/articles/clyxxr9n2kjo
[19/21] https://www

# ✅ Verify Extracted Articles

In [8]:
import json

with open(
    "../data/processed/bbc_articles.json",
    "r",
    encoding="utf-8"
) as f:

    articles = json.load(f)

print(f"Total Articles : {len(articles)}")

Total Articles : 21


In [9]:
articles[0]

{'url': 'https://www.bbc.com/news/articles/cy4e1lzwgvxo',
 'title': 'Bangladesh courts China even as ties with India improve',
 'content': 'Bangladesh\'s new government has sought greater Chinese investments and partnership to revive a stuttering economy, even as it attempts to re-balance ties with neighbouring India.\nBangladeshi Prime Minister Tarique Rahman went to Malaysia and China in his first overseas official visit last month, signalling the direction of Dhaka\'s foreign policy.\nAnalysts say the choice of destinations reflects Dhaka\'s effort to recalibrate its strategic priorities. While Rahman first visited Malaysia, his trip to China is seen as the more significant.\nIndia has historically been the traditional port of call for newly-elected South Asian leaders. Some in India have viewed Rahman\'s China visit as a message to Delhi which has maintained close ties with the ousted Bangladeshi leader Sheikh Hasina.\nAmong several bilateral agreements, Rahman\'s outreach to Beiji

In [10]:
print(articles[0]["title"])

Bangladesh courts China even as ties with India improve


In [11]:
print(articles[0]["content"][:1000])

Bangladesh's new government has sought greater Chinese investments and partnership to revive a stuttering economy, even as it attempts to re-balance ties with neighbouring India.
Bangladeshi Prime Minister Tarique Rahman went to Malaysia and China in his first overseas official visit last month, signalling the direction of Dhaka's foreign policy.
Analysts say the choice of destinations reflects Dhaka's effort to recalibrate its strategic priorities. While Rahman first visited Malaysia, his trip to China is seen as the more significant.
India has historically been the traditional port of call for newly-elected South Asian leaders. Some in India have viewed Rahman's China visit as a message to Delhi which has maintained close ties with the ousted Bangladeshi leader Sheikh Hasina.
Among several bilateral agreements, Rahman's outreach to Beijing for help managing the Teesta River and a deal to develop a special economic zone near Mongla port have attracted particular attention.
These are c

In [12]:
for article in articles[:5]:

    print(article["title"])

    print(len(article["content"]))

    print("-"*60)

Bangladesh courts China even as ties with India improve
7476
------------------------------------------------------------
India orders Meta to remove paid ads promoting child sexual abuse
3511
------------------------------------------------------------
A global hub for fake luxury goods, Vietnam tries to clean up its black market
10005
------------------------------------------------------------
Marine Le Pen appeal verdict: Why this moment matters for France
7529
------------------------------------------------------------
Australia space agency finds 'likely source' of mystery space balls on beach
2871
------------------------------------------------------------


# 📄 Step 11: Convert Articles to LangChain Documents

In [13]:
import sys
import importlib

sys.path.append("../src")

import processing.document_loader as loader

importlib.reload(loader)

documents = loader.main()

Loaded 21 documents.


In [14]:
documents[0]

Document(metadata={'title': 'Bangladesh courts China even as ties with India improve', 'url': 'https://www.bbc.com/news/articles/cy4e1lzwgvxo', 'source': 'BBC'}, page_content='Bangladesh\'s new government has sought greater Chinese investments and partnership to revive a stuttering economy, even as it attempts to re-balance ties with neighbouring India.\nBangladeshi Prime Minister Tarique Rahman went to Malaysia and China in his first overseas official visit last month, signalling the direction of Dhaka\'s foreign policy.\nAnalysts say the choice of destinations reflects Dhaka\'s effort to recalibrate its strategic priorities. While Rahman first visited Malaysia, his trip to China is seen as the more significant.\nIndia has historically been the traditional port of call for newly-elected South Asian leaders. Some in India have viewed Rahman\'s China visit as a message to Delhi which has maintained close ties with the ousted Bangladeshi leader Sheikh Hasina.\nAmong several bilateral agr

In [15]:
print(documents[0].page_content[:500])

Bangladesh's new government has sought greater Chinese investments and partnership to revive a stuttering economy, even as it attempts to re-balance ties with neighbouring India.
Bangladeshi Prime Minister Tarique Rahman went to Malaysia and China in his first overseas official visit last month, signalling the direction of Dhaka's foreign policy.
Analysts say the choice of destinations reflects Dhaka's effort to recalibrate its strategic priorities. While Rahman first visited Malaysia, his trip 


In [16]:
documents[0].metadata

{'title': 'Bangladesh courts China even as ties with India improve',
 'url': 'https://www.bbc.com/news/articles/cy4e1lzwgvxo',
 'source': 'BBC'}

In [17]:
for document in documents[:3]:

    print(document.metadata["title"])

    print(document.metadata["url"])

    print(len(document.page_content))

    print("-"*60)

Bangladesh courts China even as ties with India improve
https://www.bbc.com/news/articles/cy4e1lzwgvxo
7476
------------------------------------------------------------
India orders Meta to remove paid ads promoting child sexual abuse
https://www.bbc.com/news/articles/cql11969g2qo
3511
------------------------------------------------------------
A global hub for fake luxury goods, Vietnam tries to clean up its black market
https://www.bbc.com/news/articles/cdx7zzywz7wo
10005
------------------------------------------------------------


# ✂️ Step 12: Chunk LangChain Documents

In [18]:
import sys
import importlib

sys.path.append("../src")

import processing.chunking as chunking

importlib.reload(chunking)

chunks = chunking.main()

Loaded 21 documents.
Original Documents : 21
Total Chunks : 141


In [19]:
len(chunks)

141

In [20]:
chunks[0]

Document(metadata={'title': 'Bangladesh courts China even as ties with India improve', 'url': 'https://www.bbc.com/news/articles/cy4e1lzwgvxo', 'source': 'BBC'}, page_content="Bangladesh's new government has sought greater Chinese investments and partnership to revive a stuttering economy, even as it attempts to re-balance ties with neighbouring India.\nBangladeshi Prime Minister Tarique Rahman went to Malaysia and China in his first overseas official visit last month, signalling the direction of Dhaka's foreign policy.\nAnalysts say the choice of destinations reflects Dhaka's effort to recalibrate its strategic priorities. While Rahman first visited Malaysia, his trip to China is seen as the more significant.\nIndia has historically been the traditional port of call for newly-elected South Asian leaders. Some in India have viewed Rahman's China visit as a message to Delhi which has maintained close ties with the ousted Bangladeshi leader Sheikh Hasina.\nAmong several bilateral agreeme

In [21]:
print(chunks[0].page_content)

Bangladesh's new government has sought greater Chinese investments and partnership to revive a stuttering economy, even as it attempts to re-balance ties with neighbouring India.
Bangladeshi Prime Minister Tarique Rahman went to Malaysia and China in his first overseas official visit last month, signalling the direction of Dhaka's foreign policy.
Analysts say the choice of destinations reflects Dhaka's effort to recalibrate its strategic priorities. While Rahman first visited Malaysia, his trip to China is seen as the more significant.
India has historically been the traditional port of call for newly-elected South Asian leaders. Some in India have viewed Rahman's China visit as a message to Delhi which has maintained close ties with the ousted Bangladeshi leader Sheikh Hasina.
Among several bilateral agreements, Rahman's outreach to Beijing for help managing the Teesta River and a deal to develop a special economic zone near Mongla port have attracted particular attention.


In [22]:
chunks[0].metadata

{'title': 'Bangladesh courts China even as ties with India improve',
 'url': 'https://www.bbc.com/news/articles/cy4e1lzwgvxo',
 'source': 'BBC'}

In [23]:
import processing.document_loader as loader

documents = loader.main()

print("Documents :", len(documents))
print("Chunks :", len(chunks))

Loaded 21 documents.
Documents : 21
Chunks : 141


# 🧠 Step 13: Generate Embeddings

In [24]:
import sys
import importlib

sys.path.append("../src")

import processing.embeddings as embedding_module

importlib.reload(embedding_module)

chunks, embeddings = embedding_module.main()

Loaded 21 documents.
Original Documents : 21
Total Chunks : 141


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Dimension : 384
Chunks Ready : 141


In [25]:
len(chunks)

141

In [26]:
vector = embeddings.embed_query(
    "Machine Learning"
)

print(len(vector))

384


In [27]:
vector[:10]

[-0.02439185045659542,
 0.0032444680109620094,
 0.05426763743162155,
 -0.006672574672847986,
 0.003935667220503092,
 -0.00795737560838461,
 0.02502521686255932,
 -0.03203270211815834,
 -0.05451072379946709,
 -0.04470206797122955]

In [28]:
embeddings.embed_documents(
    [chunks[0].page_content]
)

[[0.10221650451421738,
  0.025749672204256058,
  0.06137977913022041,
  0.026209000498056412,
  0.022007914260029793,
  -0.03579571098089218,
  0.03205391764640808,
  0.0018414020305499434,
  -0.06599932163953781,
  -0.07387043535709381,
  -0.018969306722283363,
  -0.06834720075130463,
  0.008316310122609138,
  0.052772071212530136,
  0.10597460716962814,
  0.0645546093583107,
  -0.007459535263478756,
  -0.04116292670369148,
  0.02512155845761299,
  -0.0723845511674881,
  -0.07159736752510071,
  -0.014476018026471138,
  0.007870185188949108,
  -0.08508734405040741,
  -0.024849215522408485,
  -0.06729425489902496,
  0.01947936601936817,
  -0.04645724967122078,
  0.011846198700368404,
  0.011105886660516262,
  -0.006180111784487963,
  0.0909629613161087,
  -0.10175401717424393,
  0.02780221961438656,
  0.05010104179382324,
  0.07871780544519424,
  -0.0027516891714185476,
  0.062353283166885376,
  0.021541008725762367,
  -0.0489269495010376,
  0.11009501665830612,
  0.027354858815670013,


# 🗄️ Step 14: Create FAISS Vector Database

In [29]:
import sys
import importlib

sys.path.append("../src")

import processing.vector_store as vector_store_module

importlib.reload(vector_store_module)

vector_store = vector_store_module.main()

Loaded 21 documents.
Original Documents : 21
Total Chunks : 141


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Dimension : 384
Chunks Ready : 141
Vector Store saved at:
X:\Upflairs\News RAG\faiss_index
Indexed Chunks : 141


In [30]:
vector_store.index.ntotal

141

In [31]:
results = vector_store.similarity_search(

    query="Artificial Intelligence",

    k=3

)

In [32]:
for i, document in enumerate(results, 1):

    print("="*80)

    print(f"Result {i}")

    print("="*80)

    print(document.metadata["title"])

    print()

    print(document.page_content[:400])

Result 1
Marine Le Pen appeal verdict: Why this moment matters for France

The RN message for voters is one of unity, but the idea of a Plan B now appears to be so accepted that Bardella is already doing marginally better than she is in latest polls, placing them both at above 30% for the first round.
Political opponents have mocked the idea that Le Pen will simply leave Bardella alone, and they also believe she would present a far bigger threat than him in a second-roun
Result 2
Pizza Express held inquiry into Andrew Mountbatten Windsor's Woking claim

A woman who received a payout from the Met Police says reporting a predatory officer was distressing.
Timir Ahmed Mohamed is accused of attempted murder after five people were hit by a car in London.
Timir Ahmed Mohamed is accused of the attempted murder of five people in Ealing, west London.
Copyright 2026 BBC. All rights reserved. The BBC is not responsible for the content of external sites.Read
Result 3
Punjab: Why Akal Takht has obj

# 🔍 Step 15: Create Document Retriever

In [1]:
import sys
import importlib

sys.path.append("../src")

import rag.retriever as retriever_module

importlib.reload(retriever_module)

retriever = retriever_module.main()

Loaded 21 documents.
Original Documents : 21
Total Chunks : 141


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Dimension : 384
Chunks Ready : 141
Retrieved Documents : 4
Result 1
India orders Meta to remove paid ads promoting child sexual abuse
https://www.bbc.com/news/articles/cql11969g2qo

The Indian government has directed Meta to immediately disable advertisements and content on Instagram that promote or facilitate child sexual abuse material, a senior official said.
It comes after a BBC Eye investigation found that Instagram has beenrunning paid advertspromoting child sexual abuse material in India, some of which linked users to Telegram channels where the material was offered fo

Result 2
India orders Meta to remove paid ads promoting child sexual abuse
https://www.bbc.com/news/articles/cql11969g2qo

Telegram said it had removed more than 274,000 groups and channels related to child sexual abuse material in 2026.
"The government has issued a stern notice to Meta over child sexual exploitative and abuse material appearing in paid advertisements on Instagram," a senior official in

In [2]:
type(retriever)

langchain_core.vectorstores.base.VectorStoreRetriever

In [3]:
docs = retriever.invoke(
    "Latest political news"
)

In [4]:
len(docs)

4

In [5]:
for document in docs:

    print(document.metadata["title"])

Punjab: Why Akal Takht has objected to a new anti-sacrilege law
India orders Meta to remove paid ads promoting child sexual abuse
Bangladesh courts China even as ties with India improve
Marine Le Pen appeal verdict: Why this moment matters for France


In [6]:
for document in docs:

    print(document.metadata["url"])

https://www.bbc.com/news/articles/crlw1xrzd8yo
https://www.bbc.com/news/articles/cql11969g2qo
https://www.bbc.com/news/articles/cy4e1lzwgvxo
https://www.bbc.com/news/articles/clyeele4leeo


In [7]:
print(docs[0].page_content[:500])

Vice-President Sara Duterte will be banned from public office if found guilty.
The Trump administration wants Vietnam to stamp out its booming counterfeit industry. Locals are divided.
A new book explores the Fab Four's dramatic visit to the country that left them worried for their safety.
West Bengal's move to replace eggs in some school meals has sparked a wider debate over nutrition and choice.
Copyright 2026 BBC. All rights reserved. The BBC is not responsible for the content of external sit


# 🤖 Step 16: Build News RAG Chain

In [8]:
import sys
import importlib

sys.path.append("../src")

import rag.rag_chain as rag_chain_module

importlib.reload(rag_chain_module)

rag_chain = rag_chain_module.create_rag_chain()

Loaded 21 documents.
Original Documents : 21
Total Chunks : 141


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Dimension : 384
Chunks Ready : 141
Retrieved Documents : 4
Result 1
India orders Meta to remove paid ads promoting child sexual abuse
https://www.bbc.com/news/articles/cql11969g2qo

The Indian government has directed Meta to immediately disable advertisements and content on Instagram that promote or facilitate child sexual abuse material, a senior official said.
It comes after a BBC Eye investigation found that Instagram has beenrunning paid advertspromoting child sexual abuse material in India, some of which linked users to Telegram channels where the material was offered fo

Result 2
India orders Meta to remove paid ads promoting child sexual abuse
https://www.bbc.com/news/articles/cql11969g2qo

Telegram said it had removed more than 274,000 groups and channels related to child sexual abuse material in 2026.
"The government has issued a stern notice to Meta over child sexual exploitative and abuse material appearing in paid advertisements on Instagram," a senior official in

In [9]:
response = rag_chain.invoke(
    {
        "input":"What are today's top news stories?"
    }
)

print(response["answer"])

Here are some of today's top news stories: 

1. Police arrested nine people and rescued over 400 cats destined for slaughter, reuniting more than 40 with their owners.
2. The Trump administration wants Vietnam to stamp out its booming counterfeit industry, with locals being divided on the issue.
3. Vice-President Sara Duterte will be banned from public office if found guilty.
4. A social worker witnessed girls being groomed but her calls for action were unheeded.
5. Instagram showed adverts featuring adult pornography and child sexual abuse material to an alias account, prompting a response from Meta.
6. The two days of violence at Negombo Prison are the worst prison riots in the country in years.
7. Coffees at some city centre outlets now cost £5 due to tariffs, climate, Gen Z cultural tastes, and savvy coffee farmers playing the market.
8. A new book explores the Fab Four's dramatic visit to a country that left them worried for their safety.
9. West Bengal's move to replace eggs in s

In [10]:
questions = [

    "Latest AI news",

    "What happened in politics?",

    "Tell me about the economy.",

    "Any international news?"
]

for question in questions:

    print("\n")

    print("=" * 80)

    print(question)

    print("=" * 80)

    answer = rag_chain.invoke(
        {
            "input": question
        }
    )

    print(answer["answer"])



Latest AI news
I couldn't find this information in the indexed news articles.


What happened in politics?
Le Pen may be acquitted which would allow her to run for presidency with her reputation intact, although this verdict is seen as unlikely. She has been found to have "authoritatively and with determination embraced the system established by her father" and was "at the heart" of a fake-jobs scheme. Additionally, Vice-President Sara Duterte will be banned from public office if found guilty. In Australia, the government has been criticized for comments made by the Prime Minister, with the acting Prime Minister stating that the government is "utterly committed" to the elevation of women in society. The Australian government also has equality in terms of the number of men and women in cabinet. Marine Le Pen's party, Rassemblement National- National Rally (RN), achieved its best-ever election performance with 143 seats in the 577-seat National Assembly in 2024.


Tell me about the eco

In [11]:
while True:

    question = input("Ask News > ")

    if question.lower() == "exit":

        break

    response = rag_chain.invoke(
        {
            "input": question
        }
    )

    print()

    print(response["answer"])

In [17]:
import sys
import importlib

sys.path.append("../src")

import rag.rag_chain as rag_chain_module

importlib.reload(rag_chain_module)

<module 'rag.rag_chain' from 'x:\\Upflairs\\News RAG\\notebooks\\../src\\rag\\rag_chain.py'>

In [18]:
rag_chain = rag_chain_module.create_rag_chain()

Loaded 21 documents.
Original Documents : 21
Total Chunks : 141


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Dimension : 384
Chunks Ready : 141
Retrieved Documents : 4
Result 1
India orders Meta to remove paid ads promoting child sexual abuse
https://www.bbc.com/news/articles/cql11969g2qo

The Indian government has directed Meta to immediately disable advertisements and content on Instagram that promote or facilitate child sexual abuse material, a senior official said.
It comes after a BBC Eye investigation found that Instagram has beenrunning paid advertspromoting child sexual abuse material in India, some of which linked users to Telegram channels where the material was offered fo

Result 2
India orders Meta to remove paid ads promoting child sexual abuse
https://www.bbc.com/news/articles/cql11969g2qo

Telegram said it had removed more than 274,000 groups and channels related to child sexual abuse material in 2026.
"The government has issued a stern notice to Meta over child sexual exploitative and abuse material appearing in paid advertisements on Instagram," a senior official in

In [19]:
result = rag_chain_module.ask_question(
    rag_chain,
    "What happened in Ukraine?"
)

In [20]:
print(result["answer"])

There were missile strikes in Ukraine, specifically in the capital city of Kyiv, resulting in widespread destruction, injuries, and deaths. At least 15 people were killed in Kyiv and 8 more in the wider Kyiv region. The Ukrainian Air Force was unable to intercept any of the 23 ballistic missiles fired by Russia, due to a "serious shortage" of interceptor missiles. The attack consisted of 68 missiles and 351 strike drones, with the air force able to shoot down or suppress 37 missiles and 326 drones.


In [21]:
rag_chain_module.display_sources(
    result["sources"]
)


📚 Sources

• Ukraine-Russia war: Warning of interceptor missile shortage as 23 killed in Kyiv region
https://www.bbc.com/news/articles/cewqqnd7zdwo

